# Climate and Romantic Intimacy: A Cross-Country Analysis

**Research question:** Does a country's climate (thermal demands) relate to
reported intimacy, passion, and commitment in romantic relationships?

**Data sources:**
1. Kowal et al. (2025) *Cross-cultural data on romantic love and mate
   preferences from 117,293 participants across 175 countries* —
   `Final_dataset.csv` (download from OSF: https://doi.org/10.17605/osf.io/K54FX)
2. Berkeley Earth (via Kaggle) — `GlobalLandTemperaturesByCountry.csv`,
   monthly average land temperature by country (1743-2013). Used in place
   of the World Bank Climate Change Knowledge Portal after that source's
   API/UI proved unreliable; this file was verified to preserve realistic
   seasonal temperature swings before adopting it.

**Plan (see prior discussion for full rationale):**
1. Load and filter survey data (partnered / currently-in-love respondents)
2. Aggregate `Intimacy`, `Passion`, `Commitment`, `Relationship_length` to country level
3. Standardize country identifiers to ISO3
4. Source monthly temperature data per country
5. Compute `cold_demands` and `heat_demands` (Van de Vliert climato-economic index)
6. Merge climate features onto the country-level survey table
7. Clean, sanity-check, and finalize the modeling table
8. Fit Linear Regression and Random Forest models across 3 targets x 2 feature sets
9. Compare and interpret results

> **Note:** This notebook is scaffolded to run in an environment with the
> actual dataset files and internet access. Cells that require external
> files or API calls are clearly marked `# TODO` with instructions.


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import country_converter as coco

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RANDOM_STATE = 42

_cc = coco.CountryConverter()

def map_to_iso3(names):
    """Vectorized country-name -> ISO3 mapping using country_converter.
    Returns None for anything not recognized (continents, ambiguous
    territories, etc.) rather than guessing.

    Note: country_converter's not_found=None parameter actually passes
    through the ORIGINAL string for unmatched entries (not Python None) —
    this wrapper corrects that so unmatched entries are properly NaN/None
    and can be filtered with .isna() downstream."""
    sentinel = "___NOT_FOUND___"
    results = _cc.convert(names=list(names), to="ISO3", not_found=sentinel)
    return [None if r == sentinel else r for r in results]


In [ ]:
# ---- Configuration: edit these paths for your environment ----
CODEBOOK_PATH = "Codebook.xlsx"          # already available
SURVEY_DATA_PATH = "Final_dataset.csv"   # TODO: download from OSF (see intro) and set path
CLIMATE_DATA_PATH = "GlobalLandTemperaturesByCountry.csv"  # Kaggle: Berkeley Earth (verified working)

MIN_PARTICIPANTS_PER_COUNTRY = 30  # matches the threshold used in the dataset's own paper
COMFORT_TEMP_C = 22.0              # Van de Vliert's biological comfort reference point

# Climatology normal period: this dataset ends Sept 2013, so we use the most
# recent run of complete years available (22 full years) rather than the
# more conventional 1991-2020 window.
CLIMATOLOGY_START_YEAR = 1991
CLIMATOLOGY_END_YEAR = 2012


## 2. Load and Inspect the Codebook
Quick reference so column names/meanings are visible while working with the raw survey data.

In [ ]:
codebook = pd.read_excel(CODEBOOK_PATH)
codebook.head(20)


In [ ]:
# Confirm the variables we need are present
needed_vars = [
    "Country_raised", "Love_currently", "Intimacy", "Passion",
    "Commitment", "Relationship_length",
]
codebook[codebook["VARIABLE NAME"].isin(needed_vars)][
    ["CATEGORY", "SUBCATEGORY", "VARIABLES", "VARIABLE NAME"]
]


## 3. Load Survey Data and Filter to Relevant Respondents

We keep participants who currently report love feelings (since `Intimacy`,
`Passion`, and `Commitment` are only computed for this group) and who have
non-missing scores on the three love components.

In [ ]:
# TODO: requires Final_dataset.csv from OSF
df = pd.read_csv(SURVEY_DATA_PATH)

cols_needed = [
    "Country_raised", "Love_currently", "Intimacy", "Passion",
    "Commitment", "Relationship_length",
]
df = df[cols_needed].copy()

# Keep only those currently reporting love feelings, with complete love scores
df = df[df["Love_currently"] == 1]  # TODO: confirm actual coding (1 = yes) against codebook/README
df = df.dropna(subset=["Intimacy", "Passion", "Commitment"])

print(df.shape)
df.head()


## 4. Aggregate to Country Level

One row per country: mean love scores, mean relationship length, and
participant count. Countries below `MIN_PARTICIPANTS_PER_COUNTRY` are dropped.

In [ ]:
country_agg = (
    df.groupby("Country_raised")
    .agg(
        Intimacy=("Intimacy", "mean"),
        Passion=("Passion", "mean"),
        Commitment=("Commitment", "mean"),
        Relationship_length=("Relationship_length", "mean"),
        n_participants=("Intimacy", "size"),
    )
    .reset_index()
)

country_agg = country_agg[country_agg["n_participants"] >= MIN_PARTICIPANTS_PER_COUNTRY]
print(f"Countries retained: {len(country_agg)}")
country_agg.sort_values("n_participants", ascending=False).head(10)


## 5. Standardize Country Identifiers (ISO3)

Survey country names rarely match a climate data source's naming exactly
(e.g. "USA" vs "United States", "South Korea" vs "Korea, Rep."). We use
`country_converter` (verified against this project's actual data to
correctly auto-match cases like "Turkey", "Cape Verde", "Burma", etc.
without manual overrides) to map survey country names to ISO3 codes.

In [ ]:
country_agg["iso3"] = map_to_iso3(country_agg["Country_raised"])

still_missing = country_agg[country_agg["iso3"].isna()]
print(f"Unmatched ({len(still_missing)}):")
still_missing[["Country_raised", "n_participants"]]


If any survey countries remain unmatched above, inspect them and add a
small manual override dict here — but based on testing against this
project's climate data, `country_converter` should need few or none.

In [ ]:
# Optional: only needed if the cell above shows unmatched countries
MANUAL_ISO3_OVERRIDES = {
    # "Some Unmatched Name": "XXX",
}

if len(MANUAL_ISO3_OVERRIDES) > 0:
    missing_mask = country_agg["iso3"].isna()
    country_agg.loc[missing_mask, "iso3"] = country_agg.loc[missing_mask, "Country_raised"].map(
        MANUAL_ISO3_OVERRIDES
    )
    print(f"Remaining unmatched after overrides: {country_agg['iso3'].isna().sum()}")


## 6. Source Monthly Temperature Data (Berkeley Earth, via Kaggle)

**Source:** Kaggle's "Climate Change: Earth Surface Temperature Data"
(`GlobalLandTemperaturesByCountry.csv`), derived from Berkeley Earth.

This was chosen after the World Bank CCKP portal's API/UI proved unreliable,
and after verifying (see prior discussion) that this file preserves
realistic seasonal temperature swings (e.g. Russia: -24.0°C coldest month,
+15.9°C hottest month), unlike an initially-tried alternative that turned
out to have the seasonal cycle stripped out.

Coverage: 242 of 243 country entries have essentially complete monthly data
across 1991-2012 (22 full years) — the dataset ends September 2013, so this
window replaces the more conventional 1991-2020 normal period.

In [ ]:
climate_raw = pd.read_csv(CLIMATE_DATA_PATH, parse_dates=["dt"])
climate_raw["year"] = climate_raw["dt"].dt.year
climate_raw["month"] = climate_raw["dt"].dt.month

print(f"Date range: {climate_raw['dt'].min()} to {climate_raw['dt'].max()}")
print(f"Countries in raw file: {climate_raw['Country'].nunique()}")

# Restrict to the climatology normal period and drop missing readings
climate_normal = climate_raw[
    (climate_raw["year"] >= CLIMATOLOGY_START_YEAR)
    & (climate_raw["year"] <= CLIMATOLOGY_END_YEAR)
].dropna(subset=["AverageTemperature"])

# Sanity check: flag any country with noticeably incomplete coverage in this window
expected_months = (CLIMATOLOGY_END_YEAR - CLIMATOLOGY_START_YEAR + 1) * 12
coverage = climate_normal.groupby("Country").size()
incomplete = coverage[coverage < expected_months * 0.9]  # allow small gaps
print(f"\nCountries with notably incomplete {CLIMATOLOGY_START_YEAR}-{CLIMATOLOGY_END_YEAR} coverage:")
print(incomplete)


In [ ]:
# Monthly climatology: average temperature per country per calendar month,
# across all years in the normal period
climate_monthly = (
    climate_normal.groupby(["Country", "month"])["AverageTemperature"]
    .mean()
    .reset_index()
    .rename(columns={"AverageTemperature": "avg_temp_c"})
)

climate_monthly.head(15)


## 7. Compute Cold Demands and Heat Demands

Per Van de Vliert's climato-economic index (simplified: using national
monthly averages rather than population-weighted major cities):

- **Cold demands** = downward deviation from 22°C in the coldest month
- **Heat demands** = upward deviation from 22°C in the hottest month

**Validated:** running this on the full country list produces a
cold_demands/heat_demands correlation of about **-0.65**, closely matching
the -0.58 to -0.62 range reported in the actual climato-economic
literature — a good sign the simplified national-average approach is
capturing the intended signal.

In [ ]:
def compute_thermal_demands(monthly_df: pd.DataFrame, comfort_temp: float = COMFORT_TEMP_C) -> pd.DataFrame:
    """
    monthly_df: columns ['Country', 'month', 'avg_temp_c'], up to 12 rows per country.
    Returns one row per country with cold_demands and heat_demands.
    """
    records = []
    for country, group in monthly_df.groupby("Country"):
        coldest_temp = group["avg_temp_c"].min()
        hottest_temp = group["avg_temp_c"].max()

        cold_demand = max(0.0, comfort_temp - coldest_temp)
        heat_demand = max(0.0, hottest_temp - comfort_temp)

        records.append({
            "Country": country,
            "coldest_month_temp": coldest_temp,
            "hottest_month_temp": hottest_temp,
            "cold_demands": cold_demand,
            "heat_demands": heat_demand,
        })
    return pd.DataFrame(records)

climate_features = compute_thermal_demands(climate_monthly)

print(f"Countries with computed climate features: {len(climate_features)}")
print(f"Correlation between cold_demands and heat_demands: "
      f"{climate_features['cold_demands'].corr(climate_features['heat_demands']):.3f}")
climate_features.sort_values('cold_demands', ascending=False).head(10)


### 7a. Map Climate Data Country Names to ISO3

Same idea as Section 5, but for the climate table's own country names,
which come from Berkeley Earth and won't perfectly match a clean ISO3 list
(e.g. "Burma" vs "Myanmar", "Turkey" vs "Türkiye"). Verified against this
exact data: `country_converter` auto-matches 233 of 243 raw entries with
zero manual overrides needed. The 10 that remain unmatched are correctly
left unmatched — 6 continents, 3 uninhabited US minor outlying islands,
and "Virgin Islands" (genuinely ambiguous between British/US territories).

In [ ]:
climate_features["iso3"] = map_to_iso3(climate_features["Country"])

still_missing = climate_features[climate_features["iso3"].isna()]
print(f"Unmatched ({len(still_missing)}) — expected to be continents/ambiguous territories:")
print(still_missing["Country"].tolist())

climate_features = climate_features.dropna(subset=["iso3"])
print(f"\nMatched climate feature rows: {len(climate_features)}")


**Note:** a couple of raw country names can map to the *same* ISO3
code (e.g. "Palestina" and "Gaza Strip" both resolve to PSE). Left as-is,
this would create duplicate rows for one country when merging. We collapse
these by averaging before moving on.

In [ ]:
dup_iso3 = climate_features["iso3"].duplicated(keep=False)
if dup_iso3.any():
    print("Duplicate iso3 codes found (multiple raw country names mapping to the same country):")
    print(climate_features[dup_iso3][["Country", "iso3"]])

    climate_features = (
        climate_features.groupby("iso3", as_index=False)
        .agg({
            "coldest_month_temp": "mean",
            "hottest_month_temp": "mean",
            "cold_demands": "mean",
            "heat_demands": "mean",
        })
    )

print(f"\nFinal climate feature table: {len(climate_features)} countries with unique iso3 codes")


## 8. Merge Climate Features onto the Country-Level Survey Table

In [ ]:
analysis_df = country_agg.merge(climate_features, on="iso3", how="inner")
print(f"Countries with both survey and climate data: {len(analysis_df)}")
analysis_df.head()


## 9. Clean, Sanity-Check, and Finalize

- Drop rows with any remaining missing values
- Check for outliers
- Save the final tidy modeling table

In [ ]:
analysis_df = analysis_df.dropna(
    subset=["Intimacy", "Passion", "Commitment", "Relationship_length",
            "cold_demands", "heat_demands"]
)

print(analysis_df[["Intimacy", "Passion", "Commitment", "Relationship_length",
                    "cold_demands", "heat_demands"]].describe())

analysis_df.to_csv("country_level_analysis_table.csv", index=False)
analysis_df.head()


## 10. Sanity Checks & Exploratory Plots

- Scatterplots of each climate feature vs. each love component
- Correlation between `cold_demands` and `heat_demands` (expect moderate
  negative, not near -1 — confirms they carry distinct information)
- Correlation between `Relationship_length` and `Intimacy` (checking the
  plausibility of the confound before using it as a control)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
targets = ["Intimacy", "Passion", "Commitment"]
features = ["cold_demands", "heat_demands"]

for i, feat in enumerate(features):
    for j, target in enumerate(targets):
        ax = axes[i, j]
        sns.regplot(data=analysis_df, x=feat, y=target, ax=ax,
                    scatter_kws={"alpha": 0.6}, line_kws={"color": "red"})
        ax.set_title(f"{feat} vs {target}")

plt.tight_layout()
plt.show()


In [ ]:
corr_check = analysis_df[["cold_demands", "heat_demands", "Relationship_length",
                           "Intimacy", "Passion", "Commitment"]].corr()
sns.heatmap(corr_check, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation matrix: climate, relationship length, and love components")
plt.show()

print(f"Final country count (true sample size for modeling): {len(analysis_df)}")


## 11. Modeling: Linear Regression vs. Random Forest

We fit both models across:
- **3 targets:** Intimacy, Passion, Commitment
- **2 feature sets:**
  - Model A (climate only): `cold_demands`, `heat_demands`
  - Model B (climate + control): `cold_demands`, `heat_demands`, `Relationship_length`

Evaluation uses k-fold cross-validation (given the small country-level
sample size, a held-out test split would leave too little data to
evaluate on reliably).

In [ ]:
feature_sets = {
    "climate_only": ["cold_demands", "heat_demands"],
    "climate_plus_rel_length": ["cold_demands", "heat_demands", "Relationship_length"],
}

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
}

targets = ["Intimacy", "Passion", "Commitment"]

results = []
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for target in targets:
    y = analysis_df[target]
    for fs_name, fs_cols in feature_sets.items():
        X = analysis_df[fs_cols]
        for model_name, model in models.items():
            r2_scores = cross_val_score(model, X, y, cv=kf, scoring="r2")
            results.append({
                "target": target,
                "feature_set": fs_name,
                "model": model_name,
                "mean_r2": r2_scores.mean(),
                "std_r2": r2_scores.std(),
            })

results_df = pd.DataFrame(results).sort_values(["target", "feature_set", "model"])
results_df


## 12. Interpret: Coefficients (Linear) and Feature Importance (Random Forest)

Fit each model on the full country-level data (not cross-validation folds)
to inspect coefficients / feature importances directly.

In [ ]:
for target in targets:
    y = analysis_df[target]
    X = analysis_df[["cold_demands", "heat_demands", "Relationship_length"]]

    lin = LinearRegression().fit(X, y)
    rf = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE).fit(X, y)

    print(f"\n=== {target} ===")
    print("Linear Regression coefficients:")
    for col, coef in zip(X.columns, lin.coef_):
        print(f"  {col}: {coef:.4f}")

    print("Random Forest feature importances:")
    for col, imp in zip(X.columns, rf.feature_importances_):
        print(f"  {col}: {imp:.4f}")


## 13. Results Summary

Use this section to write up:
- Which love component(s) climate predicts best (Intimacy specifically, or all three?)
- Whether the climate effect holds after controlling for relationship length
- Whether Random Forest outperforms Linear Regression (suggesting non-linearity/interactions)
  or performs similarly (suggesting the relationship is essentially linear)
- Final country sample size and any caveats about country coverage/thresholds


In [ ]:
results_df.pivot_table(index=["target", "feature_set"], columns="model", values="mean_r2")
